<a href="https://colab.research.google.com/github/marcos-mansur/load-forecast/blob/main/Data_quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Objective

Verify data quality
- identify missing days
- input with day before

# load libs

In [3]:
import tensorflow as tf
import numpy as np
import pandas as pd
import pendulum
from sklearn.base import BaseEstimator, TransformerMixin


# load data 

In [18]:
def download_data(start, end):
    """load data from ONS"""

    first_year = f'https://ons-dl-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_{start}.csv'

    df_20XX = pd.read_csv(first_year, 
                        sep=';', 
                        parse_dates=['din_instante'])

    for x in range(start+1,end+1):
        df_20XX = pd.concat(objs = (df_20XX,pd.read_csv(os.path.join(f'https://ons-dl-prod-opendata.s3.amazonaws.com/',
                                                                    f'dataset/carga_energia_di/CARGA_ENERGIA_{x}.csv'), 
                            sep=';', 
                            parse_dates=['din_instante'])))
    return df_20XX


df_20XX = download_data(start=2000, end=2021)

load_col = 'val_cargaenergiamwmed'
time_col = 'din_instante'

# check missing data

filter subsystem to only "SUDESTE"

In [19]:
df_se = df_20XX[df_20XX.id_subsistema == 'SE']

Check NaN and missing days:

In [20]:
def check_dq(df):
    # check for NaN values
    nan_data = df[pd.isna(df.val_cargaenergiamwmed)].din_instante
    if len(nan_data) != 0:
        print("NaN values: \n")
        print(nan_data)
    else:
        print('No missing NaN.')
    
    # check for missing days in the series
    missing_days = pd.date_range(start = df.din_instante.iloc[0], 
                                 end= df.din_instante.iloc[-1],
                                 freq='D').difference(df.din_instante)
    if len(missing_days) != 0:
        print("\nMissing days in the series:")
        print(missing_days)
    else:
        print("\nNo missing days in the series")

check_dq(df_se)


NaN values: 

1340   2013-12-01
127    2014-02-01
395    2015-04-09
383    2016-04-05
387    2016-04-06
391    2016-04-07
395    2016-04-08
399    2016-04-09
403    2016-04-10
407    2016-04-11
411    2016-04-12
415    2016-04-13
Name: din_instante, dtype: datetime64[ns]

No missing days in the series


# data treatment

In [21]:
class Preprocessor(BaseEstimator, TransformerMixin):

  def __init__(self, regiao):
    self.regiao = regiao
    self.missing_days = []
    pass


  def fit(self, df:pd.DataFrame):
    """ Learns the missing days """
    df = df.copy()
    # filter by subsystem
    df = self.filter_subsystem(df, regiao = self.regiao)
    # saves missing days in a variable called missing_days 
    self.missing_days = df[pd.isna(df.val_cargaenergiamwmed)].din_instante
    return self 


  def transform(self, df:pd.DataFrame):
    """ Applies transformations """
    df = df.copy()
    df = self.filter_subsystem(df, regiao = self.regiao)  # filter by subsystem
    df = self.impute_nan(df)                              # impute/drop NaN values
    df = self.go_to_friday(df)        # starts the dataset at a friday - the operative week 
    df = self.parse_dates(df)         # create columns parsing the data
    df = self.drop_incomplete_week(df)    # drop last rows so to have full weeks
    self.check_dq(df)                   # prints the NaN values for load and missing days
    return df


  def go_to_friday(self,df): 
    """ go next friday = begining of the operative week"""
    df = df.copy()
    # first day in dataset
    date_time = df['din_instante'].iloc[0]
    # check if the dataset starts on a friday 
    if date_time.day_name() != 'Friday':
      # today
      dt = pendulum.datetime(date_time.year,date_time.month, date_time.day)
      # next friday - begins the operative week
      next_friday = dt.next(pendulum.FRIDAY).strftime('%Y-%m-%d')
      # df starts with the begin of operative week
      df = df[df['din_instante'] >= next_friday].reset_index(drop=True).copy()
    
    return df


  def filter_subsystem(self, df:pd.DataFrame, regiao:str):
    """ filter data by subsystem and reset index """
    df = df.copy()
    # try and execept so it doesn't crash if it's applied to an already treated dataset
    try:
      df = df[df['nom_subsistema']==regiao].reset_index().drop('index',axis=1).copy()
    except:
      pass
    # dropa columns about subsystem
    df.drop(labels=['nom_subsistema','id_subsistema'], inplace=True, axis=1,errors='ignore')
    # reset index of concatenated datasets
    df.reset_index(inplace=True,drop=True)
    return df


  def parse_dates(self, df):
    """ parse date into year, month, month day and week day  """
    df = df.copy()
    
    df['semana'] = (df.index)//7
    df['dia semana'] = df['din_instante'].dt.day_name()
    df['dia mes'] = df['din_instante'].dt.day
    df['Mes'] = df['din_instante'].dt.month
    df['ano'] = df['din_instante'].dt.year
    return df

  def drop_incomplete_week(self,df):
    """ drop incomplete week at the bottom of the dataset """
    for i in range(6):
      if df['dia semana'].tail(1).item() == 'Thursday':
        break
      else:
        df.drop(labels=df.tail(1).index, axis=0, inplace=True)

    return df
  

  def impute_nan(self, df):
    """ impute the 12 NaN values """
    df = df.copy()
    time_col = 'din_instante'
    load_col = 'val_cargaenergiamwmed'
    if len(self.missing_days) != 0:
      # If the NaN weren't already dealt with:
      if df[df[time_col] == self.missing_days.iloc[0]].val_cargaenergiamwmed.isna().item():
        # impute missing day '2013-12-01' with the load from the day before
        df.at[(df[df.din_instante == self.missing_days.iloc[0]].index.item()), 
              load_col] = df[load_col].iloc[self.missing_days.index[0] - 1]
        # impute missing day '2014-02-01' with the load from the day before
        df.at[(df[df.din_instante == self.missing_days.iloc[1]].index.item()), 
              load_col] = df[load_col].iloc[self.missing_days.index[1] - 1]
        # impute missing day '2015-04-09' with the load from the day before
        df.at[(df[df.din_instante == self.missing_days.iloc[2]].index.item()), 
              load_col] = df[load_col].iloc[self.missing_days.index[2] - 1]
        # drop days from incomplete week in 2016 - from '2016-04-01' to '2016-04-14'
        df[time_col] = pd.to_datetime(df[time_col])
        df = df.drop(axis=0, index = df[(df[time_col] >= '2016-04-01') & (df[time_col] <= '2016-04-14')].index)
    
    return df
  

  def check_dq(self,df):
    # check for NaN values
    nan_data = df[pd.isna(df.val_cargaenergiamwmed)].din_instante
    if len(nan_data) != 0:
        print("NaN values: \n")
        print(nan_data)
    else:
        print('No missing NaN.')
    
    # check for missing days in the series
    missing_days = pd.date_range(start = df.din_instante.iloc[0], 
                                 end= df.din_instante.iloc[-1],
                                 freq='D').difference(df.din_instante)
    if len(missing_days) != 0:
        print("\nMissing days in the series:")
        print(missing_days)
    else:
        print("\nNo missing days in the series")

In [25]:
pp = Preprocessor(regiao = 'SUDESTE')

df = pp.fit_transform(df_20XX)

No missing NaN.

Missing days in the series:
DatetimeIndex(['2016-04-01', '2016-04-02', '2016-04-03', '2016-04-04',
               '2016-04-05', '2016-04-06', '2016-04-07', '2016-04-08',
               '2016-04-09', '2016-04-10', '2016-04-11', '2016-04-12',
               '2016-04-13', '2016-04-14'],
              dtype='datetime64[ns]', freq=None)


In [23]:
pp.missing_days

5083   2013-12-01
5145   2014-02-01
5577   2015-04-09
5939   2016-04-05
5940   2016-04-06
5941   2016-04-07
5942   2016-04-08
5943   2016-04-09
5944   2016-04-10
5945   2016-04-11
5946   2016-04-12
5947   2016-04-13
Name: din_instante, dtype: datetime64[ns]

In [24]:
df.tail()

,din_instante,val_cargaenergiamwmed,semana,dia semana,dia mes,Mes,ano
8010,2021-12-26,33942.445250,1144,Sunday,26,12,2021
8011,2021-12-27,39736.502167,1144,Monday,27,12,2021
8012,2021-12-28,40201.573667,1144,Tuesday,28,12,2021
8013,2021-12-29,40083.157792,1144,Wednesday,29,12,2021
8014,2021-12-30,38850.805500,1144,Thursday,30,12,2021
